In [3]:
import pandas as pd
import numpy as np

# 1. Load the training data
df = pd.read_csv('Data/train.csv')

# 2. Basic Shape & Overview
print(f"Dataset Shape: {df.shape}")
print("\n--- FIRST 5 ROWS ---")
display(df.head())
df.head().to_csv('Data/first 5 rows.csv')
print("\n--- DATA TYPES & STRUCTURAL INFO ---")
df.info()

# 3. Missing Value Analysis
# We calculate exactly how much data is missing to decide between imputation vs. dropping
print("\n--- MISSING VALUE SUMMARY ---")
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df)) * 100
}).sort_values(by='Percentage (%)', ascending=False)
print(missing_summary[missing_summary['Missing Count'] > 0]) # Only show columns with missing data

# 4. Class Imbalance in Target 'Drafted'
# Understanding this helps us setup Cross-Validation (Stratified K-Fold) later
print("\n--- TARGET VARIABLE DISTRIBUTION ('Drafted') ---")
counts = df['Drafted'].value_counts()
pcts = df['Drafted'].value_counts(normalize=True) * 100
for val in counts.index:
    print(f"Class {val}: {counts[val]} samples ({pcts[val]:.2f}%)")

# 5. Outlier Detection using Z-Score > 3
# Finding data points that are more than 3 standard deviations from the mean
print("\n--- OUTLIER DETECTION (Z-Score > 3) ---")
physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
for col in physical_metrics:
    mean = df[col].mean()
    std = df[col].std()
    outliers = df[(df[col] - mean).abs() > 3 * std]
    valid_count = df[col].notnull().sum()
    print(f"{col}: Found {len(outliers)} extreme outliers out of {valid_count} valid records.")

Dataset Shape: (2781, 16)

--- FIRST 5 ROWS ---


,Id,Year,Age,School,Height,Weight,Sprint_40yd,Vertical_Jump,Bench_Press_Reps,Broad_Jump,Agility_3cone,Shuttle,Player_Type,Position_Type,Position,Drafted
0,0,2011,21.0,Lehigh,1.9050,140.160042,5.39,59.69,29.0,251.46,7.91,4.94,offense,offensive_lineman,OG,1.0
1,1,2011,24.0,Abilene Christian,1.8288,87.089735,4.31,101.60,16.0,332.74,NaN,NaN,offense,backs_receivers,WR,1.0
2,2,2018,21.0,Colorado St.,1.8542,92.986436,4.51,91.44,10.0,309.88,6.95,4.37,offense,backs_receivers,WR,1.0
3,3,2010,21.0,East Carolina,1.9304,148.778297,5.09,76.20,39.0,254.00,8.12,4.71,defense,defensive_lineman,DT,1.0
4,4,2016,21.0,California,1.8796,92.079251,4.64,78.74,NaN,281.94,7.13,4.20,offense,backs_receivers,WR,1.0



--- DATA TYPES & STRUCTURAL INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 2781 entries, 0 to 2780
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                2781 non-null   int64  
 1   Year              2781 non-null   int64  
 2   Age               2346 non-null   float64
 3   School            2781 non-null   str    
 4   Height            2781 non-null   float64
 5   Weight            2781 non-null   float64
 6   Sprint_40yd       2636 non-null   float64
 7   Vertical_Jump     2227 non-null   float64
 8   Bench_Press_Reps  2060 non-null   float64
 9   Broad_Jump        2200 non-null   float64
 10  Agility_3cone     1811 non-null   float64
 11  Shuttle           1869 non-null   float64
 12  Player_Type       2781 non-null   str    
 13  Position_Type     2781 non-null   str    
 14  Position          2781 non-null   str    
 15  Drafted           2781 non-null   float64
dtypes: float64(10),

In [4]:
# Create a copy of the dataframe to keep our original data intact
train_df = df.copy()

# 1. Features to impute
physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
                    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']

# 2. Create Missing Indicator Features (The "Skipped Drill" signal)
for col in physical_metrics:
    if train_df[col].isnull().sum() > 0:
        # Creates a column like 'Sprint_40yd_Is_Missing' (1 if missing, 0 if not)
        train_df[f'{col}_Is_Missing'] = train_df[col].isnull().astype(int)

# 3. Mathematically and Physically robust imputation: Group by Position
print("--- Imputing missing values using Position Medians ---")
for col in physical_metrics:
    if train_df[col].isnull().sum() > 0:
        # Fill missing values with the median of the player's specific Position
        train_df[col] = train_df[col].fillna(train_df.groupby('Position')[col].transform('median'))

        # If any are still missing (e.g., an entire position lacked data), fall back to global median
        if train_df[col].isnull().sum() > 0:
            train_df[col] = train_df[col].fillna(train_df[col].median())

# 4. Categorical Encoding (Preparing for LightGBM/CatBoost)
categorical_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')

# Verify the transformations
print("\n--- Remaining Missing Values ---")
print(train_df[physical_metrics].isnull().sum())

print("\n--- Data Types of Categoricals ---")
print(train_df[categorical_cols].dtypes)

print("\n--- New Missing Indicator Columns Created ---")
print([col for col in train_df.columns if 'Is_Missing' in col])

--- Imputing missing values using Position Medians ---

--- Remaining Missing Values ---
Height              0
Weight              0
Sprint_40yd         0
Vertical_Jump       0
Bench_Press_Reps    0
Broad_Jump          0
Agility_3cone       0
Shuttle             0
Age                 0
dtype: int64

--- Data Types of Categoricals ---
School           category
Player_Type      category
Position_Type    category
Position         category
dtype: object

--- New Missing Indicator Columns Created ---
['Sprint_40yd_Is_Missing', 'Vertical_Jump_Is_Missing', 'Bench_Press_Reps_Is_Missing', 'Broad_Jump_Is_Missing', 'Agility_3cone_Is_Missing', 'Shuttle_Is_Missing', 'Age_Is_Missing']


In [5]:
# 1. BMI
train_df['BMI'] = train_df['Weight'] / (train_df['Height'] ** 2)

# 2. Speed-to-Weight Ratio (Momentum Proxy)
train_df['Speed_Weight_Ratio'] = train_df['Weight'] / train_df['Sprint_40yd']

# 3. Explosiveness Index (Lower Body Power Volume)
train_df['Explosiveness_Index'] = train_df['Vertical_Jump'] * train_df['Broad_Jump']

# 4. Upper Body Power Score
train_df['Power_Score'] = train_df['Bench_Press_Reps'] * train_df['Weight']

# 5. School Prestige (Combine Count)
train_df['School_Combine_Count'] = train_df.groupby('School')['Id'].transform('count')

# Let's check the correlation of these new physical features with our Target
engineered_features = ['BMI', 'Speed_Weight_Ratio', 'Explosiveness_Index', 'Power_Score', 'School_Combine_Count', 'Drafted']
correlations = train_df[engineered_features].corr()['Drafted'].sort_values(ascending=False)

print("--- Correlation of Engineered Features with Target ---")
print(correlations)

print("\n--- Current Data Shape ---")
print(train_df.shape)

--- Correlation of Engineered Features with Target ---
Drafted                 1.000000
Speed_Weight_Ratio      0.144869
Power_Score             0.109339
Explosiveness_Index     0.103178
School_Combine_Count    0.098373
BMI                     0.067165
Name: Drafted, dtype: float64

--- Current Data Shape ---
(2781, 28)


In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare Data
# Drop 'Id' as it's a unique identifier (not predictive)
X = train_df.drop(columns=['Id', 'Drafted'])
y = train_df['Drafted']

# Identify categorical columns explicitly for CatBoost
cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']

# 2. Set up Stratified K-Fold
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to store Out-Of-Fold (OOF) predictions
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

print("--- Starting 5-Fold Cross-Validation ---")

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

    # --- LightGBM ---
    # Relies on pandas 'category' dtypes automatically
    model_lgb = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, random_state=42, verbose=-1)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    oof_lgb[valid_idx] = model_lgb.predict_proba(X_valid)[:, 1]

    # --- CatBoost ---
    # Needs explicit declaration of cat_features
    model_cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, random_seed=42, verbose=0)
    model_cat.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=cat_cols,
        early_stopping_rounds=50
    )
    oof_cat[valid_idx] = model_cat.predict_proba(X_valid)[:, 1]

    # --- XGBoost ---
    # enable_categorical=True and tree_method='hist' are required to handle pandas categories natively
    model_xgb = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.05, random_state=42,
        enable_categorical=True, tree_method='hist', early_stopping_rounds=50
    )
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )
    oof_xgb[valid_idx] = model_xgb.predict_proba(X_valid)[:, 1]

    print(f"Completed Fold {fold + 1} / {N_SPLITS}")

# 3. Calculate and Print Final Out-Of-Fold ROC-AUC Scores
print("\n--- Final Cross-Validation ROC-AUC Scores ---")
print(f"LightGBM OOF ROC-AUC: {roc_auc_score(y, oof_lgb):.5f}")
print(f"CatBoost OOF ROC-AUC: {roc_auc_score(y, oof_cat):.5f}")
print(f"XGBoost  OOF ROC-AUC: {roc_auc_score(y, oof_xgb):.5f}")

--- Starting 5-Fold Cross-Validation ---
Completed Fold 1 / 5
Completed Fold 2 / 5
Completed Fold 3 / 5
Completed Fold 4 / 5
Completed Fold 5 / 5

--- Final Cross-Validation ROC-AUC Scores ---
LightGBM OOF ROC-AUC: 0.81878
CatBoost OOF ROC-AUC: 0.83896
XGBoost  OOF ROC-AUC: 0.81728


In [9]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# --- 1. Optuna Tuning for CatBoost ---
def objective(trial):
    # Define the search space
    params = {
        'iterations': trial.suggest_int('iterations', 600, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_seed': 42,
        'verbose': 0
    }

    oof_preds = np.zeros(len(X))

    # We use the same K-Fold strategy as Step 4
    for train_idx, valid_idx in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

        model = CatBoostClassifier(**params, cat_features=cat_cols)
        model.fit(
            X_train, y_train,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=50
        )
        oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]

    return roc_auc_score(y, oof_preds)

print("--- Starting Optuna Tuning for CatBoost (10 Trials) ---")
# Using 10 trials for speed. You can increase this to 30-50 later for even better results!
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print("\n--- Best CatBoost Parameters ---")
print(study.best_params)
print(f"Best Tuned CatBoost OOF ROC-AUC: {study.best_value:.5f}")

# --- 2. Ensembling ---
print("\n--- Ensembling Base Models ---")
# We apply a weighted average to the base models from Step 4.
# Giving the most weight (60%) to our best model (CatBoost), and 20% to the others.
ensemble_oof = (0.6 * oof_cat) + (0.2 * oof_lgb) + (0.2 * oof_xgb)
ensemble_score = roc_auc_score(y, ensemble_oof)
print(f"Weighted Ensemble OOF ROC-AUC: {ensemble_score:.5f}")

[I 2026-05-21 19:31:32,241] A new study created in memory with name: no-name-36f237fe-27df-4f26-9dc9-3a396068a4e7


--- Starting Optuna Tuning for CatBoost (10 Trials) ---


[I 2026-05-21 19:32:46,669] Trial 0 finished with value: 0.8399458072038535 and parameters: {'iterations': 1025, 'learning_rate': 0.03247783638534686, 'depth': 6, 'l2_leaf_reg': 2.9971896594090532}. Best is trial 0 with value: 0.8399458072038535.
[I 2026-05-21 19:33:38,763] Trial 1 finished with value: 0.8406700035274088 and parameters: {'iterations': 766, 'learning_rate': 0.051835075993412576, 'depth': 4, 'l2_leaf_reg': 9.358762021423189}. Best is trial 1 with value: 0.8406700035274088.
[I 2026-05-21 19:35:09,281] Trial 2 finished with value: 0.8394966580352899 and parameters: {'iterations': 673, 'learning_rate': 0.017922896781997474, 'depth': 7, 'l2_leaf_reg': 5.868546012089426}. Best is trial 1 with value: 0.8406700035274088.
[I 2026-05-21 19:37:20,044] Trial 3 finished with value: 0.8392862611394097 and parameters: {'iterations': 1355, 'learning_rate': 0.010630074166405705, 'depth': 5, 'l2_leaf_reg': 4.3703907279594825}. Best is trial 1 with value: 0.8406700035274088.
[I 2026-05-21


--- Best CatBoost Parameters ---
{'iterations': 766, 'learning_rate': 0.051835075993412576, 'depth': 4, 'l2_leaf_reg': 9.358762021423189}
Best Tuned CatBoost OOF ROC-AUC: 0.84067

--- Ensembling Base Models ---
Weighted Ensemble OOF ROC-AUC: 0.83851


In [11]:
import pandas as pd
from catboost import CatBoostClassifier

print("--- Loading Test Data ---")
test_df = pd.read_csv('Data/test.csv')

# 1. Preprocessing Test Data (Matching Train exactly)
physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
                    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']

# Create missing indicators
for col in physical_metrics:
    # If it was missing in train, we need the column in test to match shapes
    if f'{col}_Is_Missing' in train_df.columns:
        test_df[f'{col}_Is_Missing'] = test_df[col].isnull().astype(int)

# Impute missing values
for col in physical_metrics:
    if test_df[col].isnull().sum() > 0:
        test_df[col] = test_df[col].fillna(test_df.groupby('Position')[col].transform('median'))
        if test_df[col].isnull().sum() > 0:
            test_df[col] = test_df[col].fillna(train_df[col].median()) # Fallback to train median

# Cast categoricals
categorical_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
for col in categorical_cols:
    test_df[col] = test_df[col].astype('category')

# 2. Feature Engineering on Test Data
print("--- Applying Feature Engineering to Test Data ---")
test_df['BMI'] = test_df['Weight'] / (test_df['Height'] ** 2)
test_df['Speed_Weight_Ratio'] = test_df['Weight'] / test_df['Sprint_40yd']
test_df['Explosiveness_Index'] = test_df['Vertical_Jump'] * test_df['Broad_Jump']
test_df['Power_Score'] = test_df['Bench_Press_Reps'] * test_df['Weight']
test_df['School_Combine_Count'] = test_df.groupby('School')['Id'].transform('count')

# Ensure test columns strictly match the training features order
X_test = test_df[X.columns]

# 3. Train Final Model on 100% of Training Data
print("--- Training Final Model on Full Training Set ---")
best_params = {
    'iterations': 766,
    'learning_rate': 0.051835075993412576,
    'depth': 4,
    'l2_leaf_reg': 9.358762021423189,
    'random_seed': 42,
    'verbose': 100 # Print progress every 100 trees
}

final_model = CatBoostClassifier(**best_params, cat_features=categorical_cols)
# X and y are the full training data we defined in Step 4
final_model.fit(X, y)

# 4. Predict and Save
print("--- Generating Predictions ---")
# Predict probabilities for class 1.0 (Drafted)
test_preds = final_model.predict_proba(X_test)[:, 1]

# Format exactly as Kaggle expects
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'Drafted': test_preds
})

submission.to_csv('Data/submission_1.csv', index=False)
print("SUCCESS: submission.csv has been saved and is ready to submit!")

--- Loading Test Data ---
--- Applying Feature Engineering to Test Data ---
--- Training Final Model on Full Training Set ---
0:	learn: 0.6645935	total: 15.6ms	remaining: 12s
100:	learn: 0.4152442	total: 2.81s	remaining: 18.5s
200:	learn: 0.3913342	total: 5.33s	remaining: 15s
300:	learn: 0.3702761	total: 7.54s	remaining: 11.6s
400:	learn: 0.3524896	total: 9.52s	remaining: 8.66s
500:	learn: 0.3383946	total: 11.4s	remaining: 6.04s
600:	learn: 0.3244233	total: 13.3s	remaining: 3.66s
700:	learn: 0.3120192	total: 15.3s	remaining: 1.42s
765:	learn: 0.3047477	total: 16.6s	remaining: 0us
--- Generating Predictions ---
SUCCESS: submission.csv has been saved and is ready to submit!
